# Sitzung 6: Flächen zwischen Kurven & Anwendungen

In diesem Notebook kannst du:
1. **Flächen zwischen zwei Kurven** berechnen und visualisieren
2. **Schnittpunkte** zweier Funktionen finden (sympy + Plot)
3. **Sachkontext: Füllproblem** — Zuflussrate → Bestand als Integral
4. **Momentan- vs. Gesamtänderung** verstehen
5. **Mittelwert einer Funktion** interaktiv erkunden

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from ipywidgets import interact, FloatSlider, Dropdown, IntSlider
from IPython.display import display, Markdown
%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['text.usetex'] = False
plt.rcParams['mathtext.fontset'] = 'cm'  # Computer Modern (LaTeX-Look)

---
## 1. Fläche zwischen zwei Kurven

Die Fläche zwischen zwei Funktionen $f$ und $g$ auf dem Intervall $[a, b]$ ist:

$$A = \int_a^b |f(x) - g(x)|\, dx$$

Wähle verschiedene Funktionenpaare und beobachte, wie die Fläche eingefärbt und berechnet wird.

In [ ]:
paare = {
    'x  und  x²': {
        'f': lambda x: x, 'g': lambda x: x**2,
        'f_label': r'$f(x) = x$', 'g_label': r'$g(x) = x^2$',
        'xlim': (-0.5, 1.5),
    },
    'x²−1  und  −x²+3': {
        'f': lambda x: x**2 - 1, 'g': lambda x: -x**2 + 3,
        'f_label': r'$f(x) = x^2 - 1$', 'g_label': r'$g(x) = -x^2 + 3$',
        'xlim': (-2.5, 2.5),
    },
    'x³  und  x': {
        'f': lambda x: x**3, 'g': lambda x: x,
        'f_label': r'$f(x) = x^3$', 'g_label': r'$g(x) = x$',
        'xlim': (-1.5, 1.5),
    },
    'eˣ  und  1': {
        'f': lambda x: np.exp(x), 'g': lambda x: np.ones_like(x),
        'f_label': r'$f(x) = e^x$', 'g_label': r'$g(x) = 1$',
        'xlim': (-1, 2.5),
    },
    'sin(x)  und  cos(x)': {
        'f': lambda x: np.sin(x), 'g': lambda x: np.cos(x),
        'f_label': r'$f(x) = \sin(x)$', 'g_label': r'$g(x) = \cos(x)$',
        'xlim': (-0.5, 2 * np.pi + 0.5),
    },
}

def flaeche_zwischen_kurven(wahl='x  und  x²'):
    info = paare[wahl]
    f, g = info['f'], info['g']
    a, b = info['xlim']
    x = np.linspace(a, b, 1000)
    y_f, y_g = f(x), g(x)

    fig, ax = plt.subplots()
    ax.plot(x, y_f, 'b-', linewidth=2, label=info['f_label'])
    ax.plot(x, y_g, 'r-', linewidth=2, label=info['g_label'])

    # Fläche einfärben: grün wo f > g, orange wo g > f
    ax.fill_between(x, y_f, y_g, where=(y_f >= y_g),
                    color='#2ecc71', alpha=0.35, label='$f \\geq g$')
    ax.fill_between(x, y_f, y_g, where=(y_f < y_g),
                    color='#e67e22', alpha=0.35, label='$g > f$')

    # Numerisch Fläche berechnen
    flaeche = np.trapz(np.abs(y_f - y_g), x)

    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=12, loc='upper left', framealpha=0.9)
    ax.set_title(rf'Fläche $\approx {flaeche:.4f}$', fontsize=14)
    plt.tight_layout()
    plt.show()

interact(
    flaeche_zwischen_kurven,
    wahl=Dropdown(options=list(paare.keys()), value='x  und  x²',
                  description='Paar:')
);

**Beobachte:**
- Die **grüne** Fläche zeigt, wo $f(x) \geq g(x)$ ist, die **orange** Fläche, wo $g(x) > f(x)$.
- Bei $x^3$ und $x$ gibt es **zwei Teilflächen** mit Vorzeichenwechsel — beide zählen positiv!
- Die Formel: $A = \int_a^b |f(x) - g(x)|\,dx$ — der **Betrag** ist entscheidend.

---
## 2. Schnittpunkte finden

Um die Integrationsgrenzen zu bestimmen, muss man zuerst die **Schnittpunkte** finden: $f(x) = g(x)$.

In [ ]:
_x = sp.Symbol('x')

schnittpunkt_paare = {
    'x  und  x²': (sp.sympify('x'), sp.sympify('x**2')),
    'x²−1  und  −x²+3': (sp.sympify('x**2 - 1'), sp.sympify('-x**2 + 3')),
    'x³  und  x': (sp.sympify('x**3'), sp.sympify('x')),
    'sin(x)  und  cos(x)': (sp.sin(_x), sp.cos(_x)),
}

def schnittpunkte_zeigen(wahl='x  und  x²'):
    f_sym, g_sym = schnittpunkt_paare[wahl]

    # Schnittpunkte berechnen
    loesungen = sp.solve(f_sym - g_sym, _x)
    loesungen_real = [s for s in loesungen if s.is_real]
    loesungen_float = sorted([float(s) for s in loesungen_real])

    display(Markdown(f"**$f(x) = {sp.latex(f_sym)}$, $\\quad g(x) = {sp.latex(g_sym)}$**"))
    display(Markdown(f"**Gleichung:** ${sp.latex(f_sym)} = {sp.latex(g_sym)}$"))
    display(Markdown(f"**Schnittpunkte:** $x = {', '.join(sp.latex(s) for s in loesungen_real)}$"))

    # Plot
    f_np = sp.lambdify(_x, f_sym, 'numpy')
    g_np = sp.lambdify(_x, g_sym, 'numpy')

    if loesungen_float:
        margin = max(1.5, (max(loesungen_float) - min(loesungen_float)) * 0.5)
        a = min(loesungen_float) - margin
        b = max(loesungen_float) + margin
    else:
        a, b = -3, 3

    x = np.linspace(a, b, 500)
    y_f = np.atleast_1d(f_np(x)).astype(float)
    y_g = np.atleast_1d(g_np(x)).astype(float)

    fig, ax = plt.subplots()
    ax.plot(x, y_f, 'b-', linewidth=2, label=f'$f(x) = {sp.latex(f_sym)}$')
    ax.plot(x, y_g, 'r-', linewidth=2, label=f'$g(x) = {sp.latex(g_sym)}$')

    for xs in loesungen_float:
        ys = float(f_sym.subs(_x, xs))
        ax.plot(xs, ys, 'ko', markersize=10, zorder=5)
        ax.annotate(f'$x = {xs:.2g}$', (xs, ys),
                    textcoords='offset points', xytext=(10, 15), fontsize=12,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=12, loc='upper left', framealpha=0.9)
    ax.set_title('Schnittpunkte', fontsize=14)
    plt.tight_layout()
    plt.show()

interact(
    schnittpunkte_zeigen,
    wahl=Dropdown(options=list(schnittpunkt_paare.keys()), value='x  und  x²',
                  description='Paar:')
);

**Vorgehen bei Flächenaufgaben:**
1. Schnittpunkte bestimmen: $f(x) = g(x)$ lösen
2. Skizze anfertigen: Welche Funktion liegt oben?
3. Integral aufteilen, falls sich die Lage ändert (Vorzeichenwechsel von $f - g$)
4. $A = \sum \left|\int_{x_i}^{x_{i+1}} (f(x) - g(x))\,dx\right|$

---
## 3. Sachkontext: Füllproblem

Wasser fließt mit der Rate $z(t) = 6t - t^2$ (Liter/Min) in einen Tank. Der Anfangsbestand ist $W_0 = 10$ Liter.

$$W(t) = W_0 + \int_0^t z(s)\,ds$$

Verschiebe den Zeitpunkt und beobachte, wie der **Bestand** (rechts) aus der **Zuflussrate** (links) entsteht.

In [ ]:
def fuellproblem(t_aktuell=3.0):
    W0 = 10  # Anfangsbestand in Litern
    z = lambda t: 6 * t - t**2  # Zuflussrate
    # Stammfunktion: Z(t) = 3t² - t³/3
    Z = lambda t: 3 * t**2 - t**3 / 3
    W = lambda t: W0 + Z(t) - Z(0)  # Bestandsfunktion

    t = np.linspace(0, 6, 300)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # --- Linker Plot: Zuflussrate ---
    ax1.plot(t, z(t), 'b-', linewidth=2, label=r'$z(t) = 6t - t^2$')
    t_fill = np.linspace(0, t_aktuell, 200)
    ax1.fill_between(t_fill, z(t_fill), color='#3498db', alpha=0.3)
    ax1.axvline(x=t_aktuell, color='red', linestyle='--', linewidth=1.5)

    zufluss_menge = Z(t_aktuell) - Z(0)
    ax1.set_title(rf'Zuflussrate — $\int_0^{{{t_aktuell:.1f}}} z(t)\,dt = {zufluss_menge:.1f}$ L',
                  fontsize=13)
    ax1.set_xlabel('Zeit $t$ (Min)', fontsize=12)
    ax1.set_ylabel('Rate (L/Min)', fontsize=12)
    ax1.legend(fontsize=12, loc='upper left', framealpha=0.9)
    ax1.set_xlim(0, 6)
    ax1.set_ylim(-2, 12)
    ax1.axhline(y=0, color='k', linewidth=0.5)
    ax1.grid(True, alpha=0.3)

    # --- Rechter Plot: Bestand ---
    ax2.plot(t, W(t), 'g-', linewidth=2.5,
             label=r'$W(t) = 10 + \int_0^t z(s)\,ds$')
    ax2.plot(t_aktuell, W(t_aktuell), 'ro', markersize=10, zorder=5)
    ax2.axhline(y=W0, color='gray', linestyle=':', linewidth=1,
                label=rf'$W_0 = {W0}$ L')

    ax2.set_title(rf'Wasserstand: $W({t_aktuell:.1f}) = {W(t_aktuell):.1f}$ L',
                  fontsize=13)
    ax2.set_xlabel('Zeit $t$ (Min)', fontsize=12)
    ax2.set_ylabel('Bestand (L)', fontsize=12)
    ax2.legend(fontsize=12, loc='upper left', framealpha=0.9)
    ax2.set_xlim(0, 6)
    ax2.set_ylim(0, 55)
    ax2.axhline(y=0, color='k', linewidth=0.5)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(
    fuellproblem,
    t_aktuell=FloatSlider(value=3.0, min=0, max=6, step=0.2,
                          description='t =',
                          style={'description_width': '30px'},
                          layout={'width': '400px'})
);

**Beobachte:**
- Die **blaue Fläche** unter der Zuflussrate (links) entspricht der **zugeflossenen Wassermenge**.
- Der **Bestand** (rechts) steigt, solange die Zuflussrate positiv ist.
- Bei $t = 3$ ist die Zuflussrate maximal — dort steigt der Bestand am steilsten ($W'(t) = z(t)$).
- Die Zuflussrate ist die **Ableitung** des Bestands: $W'(t) = z(t)$.

---
## 4. Momentan- vs. Gesamtänderung

Ein wichtiger Unterschied im Abitur:
- **Momentane Änderungsrate** $f'(t_0)$: Wie schnell ändert sich $f$ *genau jetzt* bei $t = t_0$?
- **Gesamtänderung** $\int_a^b f'(t)\,dt = f(b) - f(a)$: Wie viel hat sich $f$ *insgesamt* von $a$ bis $b$ geändert?

Verschiebe den Punkt und vergleiche beide Konzepte.

In [ ]:
def momentan_vs_gesamt(t0=2.0, t_ende=5.0):
    """Bevölkerungswachstum: r(t) = 500·e^(0.02t) Personen/Jahr."""
    r = lambda t: 500 * np.exp(0.02 * t)       # Änderungsrate
    # Stammfunktion: R(t) = 500/0.02 · e^(0.02t) = 25000 · e^(0.02t)
    R = lambda t: 25000 * np.exp(0.02 * t)

    t = np.linspace(0, 15, 300)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # --- Links: Änderungsrate mit Tangente ---
    ax1.plot(t, r(t), 'b-', linewidth=2, label=r'$r(t) = 500 \cdot e^{0{,}02t}$')

    # Momentane Rate hervorheben
    r_t0 = r(t0)
    ax1.plot(t0, r_t0, 'ro', markersize=10, zorder=5)
    ax1.annotate(rf'$r({t0:.0f}) = {r_t0:.0f}$ Pers./Jahr',
                 (t0, r_t0), textcoords='offset points', xytext=(15, 10),
                 fontsize=12, color='red',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    # Gesamtänderung als Fläche
    t_fill = np.linspace(0, t_ende, 200)
    ax1.fill_between(t_fill, r(t_fill), color='#2ecc71', alpha=0.3)
    gesamt = R(t_ende) - R(0)
    ax1.set_title('Änderungsrate (Ableitung)', fontsize=13)
    ax1.set_xlabel('Zeit $t$ (Jahre ab 2020)', fontsize=12)
    ax1.set_ylabel('Personen/Jahr', fontsize=12)
    ax1.legend(fontsize=11, loc='upper left', framealpha=0.9)
    ax1.set_xlim(0, 15)
    ax1.set_ylim(0, 750)
    ax1.grid(True, alpha=0.3)

    # --- Rechts: Zusammenfassung ---
    ax2.axis('off')
    text = (
        f"Momentane Änderungsrate bei $t = {t0:.0f}$:\n"
        f"$r({t0:.0f}) = {r_t0:.0f}$ Personen/Jahr\n\n"
        f"= Steigung der Bestandskurve bei $t = {t0:.0f}$\n"
        f"= Wie schnell wächst die Bevölkerung\n"
        f"    genau in diesem Moment?\n\n"
        f"{'─' * 40}\n\n"
        f"Gesamtänderung von $t = 0$ bis $t = {t_ende:.0f}$:\n"
        f"$\\int_0^{{{t_ende:.0f}}} r(t)\\,dt = {gesamt:.0f}$ Personen\n\n"
        f"= Grüne Fläche unter der Kurve\n"
        f"= Um wie viele Personen wächst die\n"
        f"    Bevölkerung insgesamt?"
    )
    ax2.text(0.05, 0.95, text, transform=ax2.transAxes,
             fontsize=13, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.8', facecolor='#f0f0f0', alpha=0.9))

    plt.tight_layout()
    plt.show()

interact(
    momentan_vs_gesamt,
    t0=FloatSlider(value=5.0, min=0, max=14, step=1,
                   description='t₀ =',
                   style={'description_width': '40px'},
                   layout={'width': '350px'}),
    t_ende=FloatSlider(value=10.0, min=1, max=15, step=1,
                       description='t_Ende =',
                       style={'description_width': '60px'},
                       layout={'width': '350px'}),
);

**Merke den Unterschied:**

| | Momentane Änderungsrate | Gesamtänderung |
|---|---|---|
| **Was?** | $f'(t_0)$ — ein einzelner Wert | $\int_a^b f'(t)\,dt = f(b) - f(a)$ |
| **Einheit** | Personen **pro** Jahr | Personen (ohne „pro") |
| **Geometrisch** | Steigung der Tangente | Fläche unter der Kurve |
| **Antwort auf** | „Wie schnell jetzt?" | „Wie viel insgesamt?" |

---
## 5. Mittelwert einer Funktion

Der **Mittelwert** (Durchschnittswert) einer Funktion $f$ auf $[a, b]$ ist:

$$\bar{f} = \frac{1}{b - a} \int_a^b f(x)\,dx$$

Geometrisch: Das Rechteck mit Höhe $\bar{f}$ und Breite $b - a$ hat **dieselbe Fläche** wie die Fläche unter $f$.

In [ ]:
mittelwert_funktionen = {
    'x²': {
        'f': lambda x: x**2,
        'label': r'$f(x) = x^2$',
        'xlim': (0, 4),
    },
    'sin(x)': {
        'f': lambda x: np.sin(x),
        'label': r'$f(x) = \sin(x)$',
        'xlim': (0, 2 * np.pi),
    },
    'eˣ': {
        'f': lambda x: np.exp(x),
        'label': r'$f(x) = e^x$',
        'xlim': (0, 3),
    },
    'Temperatur: 15 + 8·sin(...)': {
        'f': lambda x: 15 + 8 * np.sin(np.pi / 12 * (x - 6)),
        'label': r'$T(t) = 15 + 8\sin\!\left(\frac{\pi}{12}(t-6)\right)$',
        'xlim': (0, 24),
    },
}

def mittelwert_explorer(wahl='x²', a=0.0, b=3.0):
    info = mittelwert_funktionen[wahl]
    f = info['f']
    x_min, x_max = info['xlim']

    # Slider-Grenzen erzwingen
    a = max(a, x_min)
    b = min(b, x_max)
    if b <= a:
        b = a + 0.5

    x = np.linspace(x_min, x_max, 500)
    x_int = np.linspace(a, b, 500)

    # Mittelwert berechnen
    integral = np.trapz(f(x_int), x_int)
    mw = integral / (b - a)

    fig, ax = plt.subplots()

    # Funktion plotten
    ax.plot(x, f(x), 'b-', linewidth=2, label=info['label'])

    # Fläche unter f im Intervall
    ax.fill_between(x_int, f(x_int), color='#3498db', alpha=0.25,
                    label=r'$\int_a^b f(x)\,dx$')

    # Mittelwert als horizontale Linie
    ax.hlines(mw, a, b, colors='red', linewidth=2.5, linestyles='-',
              label=rf'$\bar{{f}} = {mw:.3f}$')
    # Rechteck mit gleicher Fläche (gestrichelt)
    ax.fill_between([a, b], [mw, mw], color='red', alpha=0.1)

    # Intervallgrenzen
    ax.axvline(x=a, color='gray', linestyle=':', linewidth=1)
    ax.axvline(x=b, color='gray', linestyle=':', linewidth=1)

    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=12, loc='upper left', framealpha=0.9)
    ax.set_title(
        rf'Mittelwert auf $[{a:.1f},\; {b:.1f}]$: '
        rf'$\bar{{f}} = \frac{{1}}{{{b-a:.1f}}} \cdot {integral:.2f} = {mw:.3f}$',
        fontsize=13)
    ax.set_xlim(x_min - 0.3, x_max + 0.3)
    plt.tight_layout()
    plt.show()

interact(
    mittelwert_explorer,
    wahl=Dropdown(options=list(mittelwert_funktionen.keys()), value='x²',
                  description='Funktion:'),
    a=FloatSlider(value=0.0, min=0, max=20, step=0.5,
                  description='a =',
                  style={'description_width': '30px'},
                  layout={'width': '350px'}),
    b=FloatSlider(value=3.0, min=0.5, max=24, step=0.5,
                  description='b =',
                  style={'description_width': '30px'},
                  layout={'width': '350px'}),
);

**Beobachte:**
- Die **rote Linie** (Mittelwert) teilt die Fläche so, dass das rote Rechteck **gleich viel Fläche** hat wie die blaue Fläche unter der Kurve.
- Probiere die **Temperatur-Funktion** aus: Stelle $a = 0$, $b = 24$ ein und lies die Durchschnittstemperatur ab.
- Beim **Sinus** über eine volle Periode $[0, 2\pi]$ ist der Mittelwert genau 0 — warum?

**Typische Abitur-Formulierung:**
> „Berechnen Sie die mittlere Temperatur im Zeitraum von 6 bis 18 Uhr."
> → $\bar{T} = \frac{1}{18 - 6} \int_6^{18} T(t)\,dt$

---
## Zusammenfassung

| Thema | Formel | Bedeutung |
|-------|--------|-----------|
| Fläche zwischen Kurven | $A = \int_a^b \|f(x) - g(x)\|\,dx$ | Betrag nicht vergessen! |
| Schnittpunkte | $f(x) = g(x)$ lösen | Integrationsgrenzen bestimmen |
| Bestand aus Rate | $F(t) = F_0 + \int_0^t f'(s)\,ds$ | HDI rückwärts |
| Momentane Rate | $f'(t_0)$ | Ein Wert, Einheit „pro Zeit" |
| Gesamtänderung | $\int_a^b f'(t)\,dt$ | Fläche, Einheit ohne „pro" |
| Mittelwert | $\bar{f} = \frac{1}{b-a}\int_a^b f(x)\,dx$ | Durchschnittliche Höhe |